In [1]:
!pwd

/home/roman/git-rudolfovo/datasets-internal/biqbin/k-cluster/scripts


In [7]:
!ls ../../../k-cluster/scripts/

Untitled.ipynb	data.py


In [201]:
import glob
import json
import scipy as sp
import os
import numpy as np
from pathlib import Path

file_path = os.path.dirname(os.path.realpath('.'))

file_path


'/home/roman/git-rudolfovo/datasets-internal/biqbin/k-cluster'

In [33]:
import importlib.util

spec=importlib.util.spec_from_file_location("k-cluster-original-data","../../../k-cluster/scripts/data.py")
# creates a new module based on spec
k_cluster = importlib.util.module_from_spec(spec)

# executes the module in its own namespace
# when a module is imported or reloaded.
spec.loader.exec_module(k_cluster)



In [83]:
!ls /home/roman/git-rudolfovo/datasets-internal/biqbin/k-cluster/../../k-cluster/k-cluster_40/

k-cluster_40.tar.gz	 kcluster40_050_10_1.dat  kcluster40_075_10_2.dat
kcluster40_025_10_1.dat  kcluster40_050_10_2.dat  kcluster40_075_10_3.dat
kcluster40_025_10_2.dat  kcluster40_050_10_3.dat  kcluster40_075_10_4.dat
kcluster40_025_10_3.dat  kcluster40_050_10_4.dat  kcluster40_075_10_5.dat
kcluster40_025_10_4.dat  kcluster40_050_10_5.dat  kcluster40_075_20_1.dat
kcluster40_025_10_5.dat  kcluster40_050_20_1.dat  kcluster40_075_20_2.dat
kcluster40_025_20_1.dat  kcluster40_050_20_2.dat  kcluster40_075_20_3.dat
kcluster40_025_20_2.dat  kcluster40_050_20_3.dat  kcluster40_075_20_4.dat
kcluster40_025_20_3.dat  kcluster40_050_20_4.dat  kcluster40_075_20_5.dat
kcluster40_025_20_4.dat  kcluster40_050_20_5.dat  kcluster40_075_30_1.dat
kcluster40_025_20_5.dat  kcluster40_050_30_1.dat  kcluster40_075_30_2.dat
kcluster40_025_30_1.dat  kcluster40_050_30_2.dat  kcluster40_075_30_3.dat
kcluster40_025_30_2.dat  kcluster40_050_30_3.dat  kcluster40_075_30_4.dat
kcluster40_025_30_3.dat  kcluster40_050_30

In [34]:
k_cluster.get_A

<function k-cluster-original-data.get_A(filename)>

In [91]:
def get_solutions():
    file_base = '{}/../../k-cluster/k-cluster_{}/{}.dat'
    solutions = k_cluster.get_solutions()
    sizes = [40, 80, 100, 120, 140, 160]
    for n in sizes:
        instances = solutions[f'instance_{n}'].tolist()
        optimums = solutions[f'optimum_{n}'].tolist()

        for instance, optimum in zip(instances, optimums):
            yield instance, {'optimum': optimum, 'file': file_base.format(file_path, n, instance)}


solutions = dict(get_solutions())


In [93]:
solutions

{'kcluster40_025_10_1': {'optimum': 29,
  'file': '/home/roman/git-rudolfovo/datasets-internal/biqbin/k-cluster/../../k-cluster/k-cluster_40/kcluster40_025_10_1.dat'},
 'kcluster40_025_10_2': {'optimum': 30,
  'file': '/home/roman/git-rudolfovo/datasets-internal/biqbin/k-cluster/../../k-cluster/k-cluster_40/kcluster40_025_10_2.dat'},
 'kcluster40_025_10_3': {'optimum': 27,
  'file': '/home/roman/git-rudolfovo/datasets-internal/biqbin/k-cluster/../../k-cluster/k-cluster_40/kcluster40_025_10_3.dat'},
 'kcluster40_025_10_4': {'optimum': 27,
  'file': '/home/roman/git-rudolfovo/datasets-internal/biqbin/k-cluster/../../k-cluster/k-cluster_40/kcluster40_025_10_4.dat'},
 'kcluster40_025_10_5': {'optimum': 26,
  'file': '/home/roman/git-rudolfovo/datasets-internal/biqbin/k-cluster/../../k-cluster/k-cluster_40/kcluster40_025_10_5.dat'},
 'kcluster40_025_20_1': {'optimum': 77,
  'file': '/home/roman/git-rudolfovo/datasets-internal/biqbin/k-cluster/../../k-cluster/k-cluster_40/kcluster40_025_20_1

In [261]:
sizes = [40]
folder_base = "../{}/{}/*.json"

def files(penalty_type, n):
    folder = folder_base.format(penalty_type, n)  
    for i in glob.glob(folder):
        yield i



#n = 160
for n in [40, 80, 100, 120, 140, 160]:
    input_files = files("exact", n)
    out_folder = '../1/{}/'
    #input_files = files("exact_rk_penalties", n)
    #out_folder = '../2/{}/'
    #input_files = files("exact_dp_penalties", n)
    #out_folder = '../3/{}/'
    
    
    for i in input_files:
        with open(i) as f:
            out_folder = out_folder.format(n)
            Path(out_folder).mkdir(parents=True, exist_ok=True)
    
            print(i)
            data = json.load(f)
            instance = data['instance']
            print(instance)
            l = data['lambda']
            mu = data['mu']
            optimum = data['optimum']
            target_cardinality = data['target_cardinality']
            Q = sp.sparse.coo_matrix((data['qubo']['data'], (data['qubo']['row'], data['qubo']['col'])), shape=data['qubo']['shape']).todense()
            _, A, k = k_cluster.get_A(solutions[instance]['file'])
            optimum_original = solutions[instance]['optimum']
            print(solutions[instance]['file'])
            QQ =(Q+Q.T)/2/2
            AA = (1-A)/2
            np.fill_diagonal(AA, 0)
            np.fill_diagonal(AA, -l-2*k*mu)
            AA += mu
    #        print(k, l, mu)
    #        print(QQ)
    #        print()
    #        print(AA)
            try:
                assert np.array_equal(QQ, AA)
            except AssertionError:
                print(k, l, mu)
                print(QQ)
                print()
                print(AA)
                raise AssertionError
    
            QQ = np.triu(QQ) + np.tril(QQ, -1).T
            offset = l*k + mu*k**2
            print(k, l, mu, offset)
            print(QQ)
            Q = sp.sparse.coo_matrix(QQ)
            Q.eliminate_zeros()
            data['qubo']['nnz'] = Q.nnz
            data['qubo']['data'] = Q.data.tolist()
            data['qubo']['row'] = Q.row.tolist()
            data['qubo']['col'] = Q.col.tolist()
            data['qubo']['shape'] = Q.shape
            data['offset'] = float(offset)
    
            del data['optimum']
            del data['instance']
            del data['lambda']
            del data['mu']
            del data['target_cardinality']
    
            data['qubo']["info"] = "Q: in sparse coo fromat, offset: scalar. (((Q.data, (Q.row, Q.col), Q.shape))"
            data["info"] = {
            "description": "Problem on complement of a graph for densest k-subgraph problem.I.e. sparsest k-subgraph problem. Objective is modified as follows: x^TQx = 1/2*x^TA(complement(G))x - lambda*(sum(x)-k) + mu*(sum(x) - k)^2 Where: A is adjacency matrix, offset = lambda*k + mu*k^2",
            "references": [
                "https://cedric.cnam.fr/~lamberta/Library/k-cluster.html",
                "https://cedric.cnam.fr/~lamberta/Library/solutions_k-cluster.html"
            ],
            "k": int(k),
            "lambda": l,
            "mu": mu,
            "instance": instance,
            "data_transformed_from": i
               
        }        
    
            with open(f'{out_folder}/{instance}.json', 'w') as f:
                json.dump(data, f, indent=4)
            
            #break
            
            #print(k_original, target_cardinality, optimum, optimum_original)
            


../exact/40/kcluster40_050_10_4_mu_0.json
kcluster40_050_10_4
/home/roman/git-rudolfovo/datasets-internal/biqbin/k-cluster/../../k-cluster/k-cluster_40/kcluster40_050_10_4.dat
10 2.5 0.0 25.0
[[-2.5  0.   0.  ...  1.   1.   1. ]
 [ 0.  -2.5  1.  ...  0.   0.   1. ]
 [ 0.   0.  -2.5 ...  0.   1.   0. ]
 ...
 [ 0.   0.   0.  ... -2.5  1.   1. ]
 [ 0.   0.   0.  ...  0.  -2.5  1. ]
 [ 0.   0.   0.  ...  0.   0.  -2.5]]
../exact/40/kcluster40_075_20_2_mu_0.json
kcluster40_075_20_2
/home/roman/git-rudolfovo/datasets-internal/biqbin/k-cluster/../../k-cluster/k-cluster_40/kcluster40_075_20_2.dat
20 3.5 0.0 70.0
[[-3.5  1.   1.  ...  1.   1.   0. ]
 [ 0.  -3.5  0.  ...  1.   0.   0. ]
 [ 0.   0.  -3.5 ...  1.   1.   0. ]
 ...
 [ 0.   0.   0.  ... -3.5  0.   0. ]
 [ 0.   0.   0.  ...  0.  -3.5  0. ]
 [ 0.   0.   0.  ...  0.   0.  -3.5]]
../exact/40/kcluster40_075_30_4_mu_0.json
kcluster40_075_30_4
/home/roman/git-rudolfovo/datasets-internal/biqbin/k-cluster/../../k-cluster/k-cluster_40/kcluster

In [235]:
data.keys()

dict_keys(['qubo', 'offset', 'info'])

In [262]:
data['info']

{'description': 'Problem on complement of a graph for densest k-subgraph problem.I.e. sparsest k-subgraph problem. Objective is modified as follows: x^TQx = 1/2*x^TA(complement(G))x - lambda*(sum(x)-k) + mu*(sum(x) - k)^2 Where: A is adjacency matrix, offset = lambda*k + mu*k^2',
 'references': ['https://cedric.cnam.fr/~lamberta/Library/k-cluster.html',
  'https://cedric.cnam.fr/~lamberta/Library/solutions_k-cluster.html'],
 'k': 80,
 'lambda': 19.0,
 'mu': 1.0,
 'instance': 'kcluster160_075_80_5',
 'data_transformed_from': '../exact/160/kcluster160_075_80_5_mu_nz.json'}

In [237]:
data['offset']

27000.0

In [255]:
!pwd

/home/roman/git-rudolfovo/datasets-internal/biqbin/k-cluster/scripts


In [361]:
t = 1
files = glob.glob(f'../{t}/*/*.json')


for i in files:
    with open(i) as f:
        data = json.load(f)
        data_transformed_from = data['info']['data_transformed_from'].replace('../', '')
        instance = data['info']['instance']

    res_file = f'../results/{data_transformed_from}.output.json'

    try:
        with open(res_file) as f:
            result = json.load(f)
        print('Found', res_file)
    except:
        print(res_file, 'not found')
        continue

    Q = sp.sparse.coo_matrix((data['qubo']['data'], (data['qubo']['row'], data['qubo']['col'])), shape=data['qubo']['shape']).todense().A
    _, A, k = k_cluster.get_A(solutions[instance]['file'])
    optimum_original = solutions[instance]['optimum']
    offset = data['offset']
    optimum = k*(k-1)/2 - optimum_original

    x = np.asarray(result['qubo']['x'])
    computed_val = result['qubo']['computed_val']/2 + offset

    val = Q.dot(x).dot(x) + offset

    assert optimum == val
    assert computed_val == optimum

    result['info'] = data['info']
    del result['maxcut']
    del result['qubo']['solution']
    del result['info']['data_transformed_from']
    result['qubo']['computed_val'] = computed_val
    result['qubo']['offset'] = offset
    result['qubo']['info'] = "computed_val = min(Q)+offset"
    result['info']['input_data'] = i
    result['info']['is_optimal'] = bool((computed_val == optimum))

    n, _ = Q.shape

    out_folder = f'../results/{t}/{n}'
    Path(out_folder).mkdir(parents=True, exist_ok=True)

    with open(f'{out_folder}/{instance}.output.json', 'w') as f:
        json.dump(result, f, indent=4)
    
    #break
    #print(i)
    #print(res_file)
    #print(val, computed_val, optimum, optimum_original, offset)
    #break



Found ../results/exact/80/kcluster80_075_60_2_mu_nz.json.output.json
Found ../results/exact/80/kcluster80_075_20_5_mu_0.json.output.json
Found ../results/exact/80/kcluster80_050_60_1_mu_0.json.output.json
Found ../results/exact/80/kcluster80_050_60_3_mu_0.json.output.json
Found ../results/exact/80/kcluster80_025_40_1_mu_nz.json.output.json
Found ../results/exact/80/kcluster80_025_60_5_mu_0.json.output.json
Found ../results/exact/80/kcluster80_025_20_1_mu_nz.json.output.json
Found ../results/exact/80/kcluster80_025_20_5_mu_nz.json.output.json
Found ../results/exact/80/kcluster80_025_60_4_mu_0.json.output.json
Found ../results/exact/80/kcluster80_050_40_2_mu_nz.json.output.json
Found ../results/exact/80/kcluster80_075_40_3_mu_nz.json.output.json
Found ../results/exact/80/kcluster80_075_60_3_mu_nz.json.output.json
Found ../results/exact/80/kcluster80_025_20_4_mu_0.json.output.json
Found ../results/exact/80/kcluster80_050_40_5_mu_0.json.output.json
Found ../results/exact/80/kcluster80_050_

In [434]:
import operator as pyoperator

_operators = {'<=': pyoperator.le, '>=': pyoperator.ge, '==': pyoperator.eq}

c = 'linear'
#c = 'quadratic'
t = 4
files = glob.glob(f'../../../test_instances/json/qbo_constrained/{c}/{t}/*/*.json')

for i in files:
    with open(i) as f:
        data = json.load(f)

    instance = data['instance']
    optimum = data['optimum']
    x = np.array(data['x'])
    ((Q_data, (Q_row, Q_col)), Q_shape) = data['QBO']['Q']
    Q = sp.sparse.coo_matrix((Q_data, (Q_row, Q_col)), Q_shape)
    k = data['info']['k']
    l = data['info']['lambda']
    mu = data['info']['mu']
    print()
    print(optimum, k*(k-1)/2-optimum)
    print(k, l, mu)

    

    linear = [
        ( sp.sparse.coo_matrix((Bi_data, (Bi_row, Bi_col)), shape=Bi_shape), 
         Ci, 
         sense) 
        for ((Bi_data, (Bi_row, Bi_col)), Bi_shape), Ci, sense in  data['QBO']['constraints']['linear']]

    quadratic = [
        ( sp.sparse.coo_matrix((Qi_data, (Qi_row, Qi_col)), shape=Qi_shape), 
         ri, 
         sense) 
        for ((Qi_data, (Qi_row, Qi_col)), Qi_shape), ri, sense in  data['QBO']['constraints']['quadratic']]



    for Bi, ci, sense in linear:
        Bi_x = Bi @ x
        print('linear', Bi.shape, Bi_x, sense, ci)
        for lhs, rhs in zip(Bi_x, ci):
            assert _operators[sense](lhs, rhs) 

    for Qi, ri, sense in quadratic:
        Qi_x = x.T @ Qi @ x
        print('quadratic', Qi_x, sense, ri)
        assert _operators[sense](Qi_x, ri) 
    #break

    


360.0 1410.0
60 0.0 60.0
linear (30, 80) [113 118 131 127 132 113 127 119 113 124 142 141 115 125 114 126 105 120
 134 120 132 116 129 133 122 125 121 124 125 110] >= [113, 118, 131, 127, 132, 113, 127, 119, 113, 124, 142, 141, 115, 125, 114, 126, 105, 120, 134, 120, 132, 116, 129, 133, 122, 125, 121, 124, 125, 110]

7.0 183.0
20 0.0 10.0
linear (10, 80) [49 52 38 23 33 31 34 30 42 48] >= [49, 52, 38, 23, 33, 31, 34, 30, 42, 48]

802.0 968.0
60 0.0 60.0
linear (30, 80) [127 143 110 137 115 115 116 128 120 124 115 118 138 118 107 116 130 139
 126 116 111 100 116 108 117 111 127 114 112 113] >= [127, 143, 110, 137, 115, 115, 116, 128, 120, 124, 115, 118, 138, 118, 107, 116, 130, 139, 126, 116, 111, 100, 116, 108, 117, 111, 127, 114, 112, 113]

798.0 972.0
60 0.0 60.0
linear (30, 80) [115 120 113 123 119 119 128 125 124 128  81 133 111 132 129 122 118 114
 115 129 111 121 111 130 123 121 130 124 122 107] >= [115, 120, 113, 123, 119, 119, 128, 125, 124, 128, 81, 133, 111, 132, 129, 122, 1

In [406]:
quadratic:
1: >=, Qi = A complement ?, l1 ?
2: <=, Qi = A_complement, l1
3: >=,  Q1 = A, l3
3: <=,  Q1 = A, l3




{'version': 'v1',
 'description': 'Problem on complement of a graph for densest k-subgraph problem.I.e. sparsest k-subgraph problem. Objective is modified as follows: x^TQx = 1/2*x^TA(complement(G))x - lambda*(sum(x)-k) + mu*(sum(x) - k)^2 Where: A is adjacency matrix, offset = lambda*k + mu*k^2',
 'references': ['https://cedric.cnam.fr/~lamberta/Library/k-cluster.html',
  'https://cedric.cnam.fr/~lamberta/Library/solutions_k-cluster.html'],
 'k': 70,
 'lambda': 0.0,
 'mu': 70.0}

In [389]:
 ((Bi_data, (Bi_row, Bi_col)), Bi_shape), Ci, sense = data['QBO']['constraints']['linear'][0]

In [278]:
!ls  ../results/exact/140/kcluster140_050_70_4_mu_nz.json

ls: cannot access '../results/exact/140/kcluster140_050_70_4_mu_nz.json': No such file or directory


In [ ]:
mu(sumx - k)^2
mu(sumx^2 - 2*sumx*k + k^2)

(a+b)(a+b) = aa + bb + 2ab

In [151]:
l

2.5

In [152]:
mu

0.0

In [153]:
np.triu(1-A, 1).sum(axis=1)

array([25., 22., 15., 15., 17., 14., 24., 17., 14., 13., 17., 12., 13.,
       12., 12., 11., 12.,  6., 11.,  6.,  8., 10.,  9.,  7.,  8.,  7.,
        6.,  8.,  7.,  5.,  5.,  3.,  4.,  2.,  2.,  3.,  2.,  2.,  1.,
        0.])

In [154]:
np.triu(Q/2, 1).sum(axis=1)

array([25., 22., 15., 15., 17., 14., 24., 17., 14., 13., 17., 12., 13.,
       12., 12., 11., 12.,  6., 11.,  6.,  8., 10.,  9.,  7.,  8.,  7.,
        6.,  8.,  7.,  5.,  5.,  3.,  4.,  2.,  2.,  3.,  2.,  2.,  1.,
        0.])

In [132]:
A

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.]])

In [144]:
A.shape

(40, 40)

In [63]:
mu

0.0